# Package

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

import plotly.graph_objects as go
from utilsforecast.plotting import plot_series

ROOT = Path.cwd()  # mets ici le dossier où se trouvent tes .pkl (souvent le dossier du notebook)
AR1_BUNDLE = ROOT / "AR1_h12_oos_bundle.pkl"
ARP_BUNDLE = ROOT / "ARp_h12_oos_bundle.pkl"
LR_BUNDLE  = ROOT / "linear_regression.pkl"

SERIES_ID = "UNRATE"

print("ROOT:", ROOT.resolve())
print("Exists AR1:", AR1_BUNDLE.exists())
print("Exists ARp:", ARP_BUNDLE.exists())
print("Exists LR :", LR_BUNDLE.exists())

ROOT: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook
Exists AR1: True
Exists ARp: True
Exists LR : True


In [2]:
def load_bundle(path: Path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        return obj
    return {"_raw": obj}

def find_df_in_bundle(bundle: dict):
    # cherche un DataFrame évident dans le dict
    for k, v in bundle.items():
        if isinstance(v, pd.DataFrame):
            return v, k
    # parfois c'est stocké sous "oos" / "df_oos" / "forecasts" etc.
    for k in ["df_oos", "oos", "oos_df", "forecasts", "backtest", "bkt_df"]:
        v = bundle.get(k)
        if isinstance(v, pd.DataFrame):
            return v, k
    raise KeyError(f"Aucun DataFrame trouvé dans bundle. Clés dispo: {list(bundle.keys())}")

b_ar1 = load_bundle(AR1_BUNDLE)
b_arp = load_bundle(ARP_BUNDLE)
b_lr  = load_bundle(LR_BUNDLE)

df_ar1, k1 = find_df_in_bundle(b_ar1)
df_arp, k2 = find_df_in_bundle(b_arp)
df_lr , k3 = find_df_in_bundle(b_lr)

print("AR1 df key:", k1, "| shape:", df_ar1.shape)
print("ARp df key:", k2, "| shape:", df_arp.shape)
print("LR  df key:", k3, "| shape:", df_lr.shape)

AR1 df key: oos_predictions | shape: (741, 6)
ARp df key: oos_predictions | shape: (741, 7)
LR  df key: oos_predictions | shape: (741, 8)


# Présentation

In [3]:
def ensure_date_col(df):
    out = df.copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
        return out
    if "ds" in out.columns:
        out = out.rename(columns={"ds": "date"})
        out["date"] = pd.to_datetime(out["date"])
        return out
    # sinon index = date
    out = out.reset_index().rename(columns={"index": "date"})
    out["date"] = pd.to_datetime(out["date"])
    return out

def standardize_ar1(df):
    d = ensure_date_col(df)

    # y obs
    if "y_true" in d.columns:
        yobs = "y_true"
    elif "y_obs" in d.columns:
        yobs = "y_obs"
    else:
        raise KeyError("AR1: colonne y obs introuvable (y_true ou y_obs).")

    # point forecast
    if "y_hat" in d.columns:
        yh = "y_hat"
    elif "y_pred" in d.columns:
        yh = "y_pred"
    else:
        raise KeyError("AR1: colonne forecast introuvable (y_hat ou y_pred).")

    # interval (priorité conformal lo/hi)
    if "y_lo_95" in d.columns and "y_hi_95" in d.columns:
        lo, hi = "y_lo_95", "y_hi_95"
    elif "y_hat_p05" in d.columns and "y_hat_p95" in d.columns:
        lo, hi = "y_hat_p05", "y_hat_p95"
    else:
        lo, hi = None, None

    cols = ["date", yobs, yh] + ([lo, hi] if lo else [])
    out = d[cols].rename(columns={
        yobs: "y_obs",
        yh: "AR1",
        **({lo: "AR1-lo-95", hi: "AR1-hi-95"} if lo else {})
    })
    return out

def standardize_arp(df):
    d = ensure_date_col(df)

    if "y_true" in d.columns:
        yobs = "y_true"
    elif "y_obs" in d.columns:
        yobs = "y_obs"
    else:
        raise KeyError("ARp: colonne y obs introuvable (y_true ou y_obs).")

    if "y_hat_ar" in d.columns:
        yh = "y_hat_ar"
    elif "y_hat" in d.columns:
        yh = "y_hat"
    elif "y_pred" in d.columns:
        yh = "y_pred"
    else:
        raise KeyError("ARp: colonne forecast introuvable (y_hat_ar / y_hat / y_pred).")

    if "y_hat_ar_lo_95" in d.columns and "y_hat_ar_hi_95" in d.columns:
        lo, hi = "y_hat_ar_lo_95", "y_hat_ar_hi_95"
    elif "y_lo_95" in d.columns and "y_hi_95" in d.columns:
        lo, hi = "y_lo_95", "y_hi_95"
    elif "y_hat_p05" in d.columns and "y_hat_p95" in d.columns:
        lo, hi = "y_hat_p05", "y_hat_p95"
    else:
        lo, hi = None, None

    cols = ["date", yobs, yh] + ([lo, hi] if lo else [])
    out = d[cols].rename(columns={
        yobs: "y_obs",
        yh: "ARp",
        **({lo: "ARp-lo-95", hi: "ARp-hi-95"} if lo else {})
    })
    return out

def standardize_lr(df):
    d = ensure_date_col(df)

    # si df_lr_forecasts standardisé (series_id/date/y_obs/y_hat_lr...)
    if "series_id" in d.columns:
        d = d.query("series_id == @SERIES_ID").copy()

    if "y_obs" in d.columns:
        yobs = "y_obs"
    elif "y_true" in d.columns:
        yobs = "y_true"
    else:
        # fallback : parfois "y"
        yobs = "y" if "y" in d.columns else None
    if yobs is None:
        raise KeyError("LR: colonne y obs introuvable (y_obs/y_true/y).")

    if "y_hat_lr" in d.columns:
        yh = "y_hat_lr"
    elif "y_pred" in d.columns:
        yh = "y_pred"
    elif "LinearRegression" in d.columns:
        yh = "LinearRegression"
    else:
        raise KeyError("LR: colonne forecast introuvable (y_hat_lr/y_pred/LinearRegression).")

    # interval
    if "y_hat_lr_lo_95" in d.columns and "y_hat_lr_hi_95" in d.columns:
        lo, hi = "y_hat_lr_lo_95", "y_hat_lr_hi_95"
    elif "y_lo_95" in d.columns and "y_hi_95" in d.columns:
        lo, hi = "y_lo_95", "y_hi_95"
    else:
        lo, hi = None, None

    cols = ["date", yobs, yh] + ([lo, hi] if lo else [])
    out = d[cols].rename(columns={
        yobs: "y_obs",
        yh: "LR",
        **({lo: "LR-lo-95", hi: "LR-hi-95"} if lo else {})
    })
    return out

df_ar1_std = standardize_ar1(df_ar1)
df_arp_std = standardize_arp(df_arp)
df_lr_std  = standardize_lr(df_lr)

print(df_ar1_std.head(2))
print(df_arp_std.head(2))
print(df_lr_std.head(2))


        date  y_obs       AR1  AR1-lo-95  AR1-hi-95
0 1963-12-01    0.0  0.070473        NaN        NaN
1 1964-01-01   -0.1  0.017682        NaN        NaN
        date  y_obs       ARp  ARp-lo-95  ARp-hi-95
0 1963-12-01    0.0  0.070473        NaN        NaN
1 1964-01-01   -0.1  0.017682        NaN        NaN
        date  y_obs        LR  LR-lo-95  LR-hi-95
0 1963-12-01    0.0 -0.716835       NaN       NaN
1 1964-01-01   -0.1 -0.244223       NaN       NaN


In [5]:
import pandas as pd

# =========================
# Pré-requis : df_ar1, df_arp, df_lr existent
# Chaque df doit contenir au moins une date (date/ds/index)
# =========================

def ensure_date(df):
    d = df.copy()
    if "date" in d.columns:
        d["date"] = pd.to_datetime(d["date"])
    elif "ds" in d.columns:
        d = d.rename(columns={"ds": "date"})
        d["date"] = pd.to_datetime(d["date"])
    else:
        d = d.reset_index().rename(columns={"index": "date"})
        d["date"] = pd.to_datetime(d["date"])
    return d

# =========================
# ---- AR1 ----
# =========================
df_ar1 = ensure_date(df_ar1)

ar1_map = {}
if "y_true" in df_ar1.columns: ar1_map["y_true"] = "y_obs"
elif "y_obs" in df_ar1.columns: ar1_map["y_obs"] = "y_obs"

if "y_hat" in df_ar1.columns: ar1_map["y_hat"] = "AR1"
elif "y_pred" in df_ar1.columns: ar1_map["y_pred"] = "AR1"

# intervalles 95%
if "y_lo_95" in df_ar1.columns and "y_hi_95" in df_ar1.columns:
    ar1_map["y_lo_95"] = "AR1-lo-95"
    ar1_map["y_hi_95"] = "AR1-hi-95"
elif "y_hat_p05" in df_ar1.columns and "y_hat_p95" in df_ar1.columns:
    ar1_map["y_hat_p05"] = "AR1-lo-95"
    ar1_map["y_hat_p95"] = "AR1-hi-95"

df_ar1_std = df_ar1[["date"] + list(ar1_map.keys())].rename(columns=ar1_map)

# =========================
# ---- ARp ----
# =========================
df_arp = ensure_date(df_arp)

arp_map = {}
if "y_true" in df_arp.columns: arp_map["y_true"] = "y_obs"
elif "y_obs" in df_arp.columns: arp_map["y_obs"] = "y_obs"

if "y_hat_ar" in df_arp.columns: arp_map["y_hat_ar"] = "ARp"
elif "y_hat" in df_arp.columns: arp_map["y_hat"] = "ARp"
elif "y_pred" in df_arp.columns: arp_map["y_pred"] = "ARp"

if "y_hat_ar_lo_95" in df_arp.columns and "y_hat_ar_hi_95" in df_arp.columns:
    arp_map["y_hat_ar_lo_95"] = "ARp-lo-95"
    arp_map["y_hat_ar_hi_95"] = "ARp-hi-95"
elif "y_lo_95" in df_arp.columns and "y_hi_95" in df_arp.columns:
    arp_map["y_lo_95"] = "ARp-lo-95"
    arp_map["y_hi_95"] = "ARp-hi-95"
elif "y_hat_p05" in df_arp.columns and "y_hat_p95" in df_arp.columns:
    arp_map["y_hat_p05"] = "ARp-lo-95"
    arp_map["y_hat_p95"] = "ARp-hi-95"

df_arp_std = df_arp[["date"] + list(arp_map.keys())].rename(columns=arp_map)

# =========================
# ---- Linear Regression ----
# =========================
df_lr = ensure_date(df_lr)

cols = df_lr.columns.tolist()

y_obs = next(c for c in ["y_obs","y_true","y","true"] if c in cols)
y_hat = next(c for c in ["y_hat_lr","y_pred","LinearRegression","LR"] if c in cols)
lo    = next((c for c in ["y_hat_lr_lo_95","y_lo_95","LinearRegression-lo-95","LR-lo-95"] if c in cols), None)
hi    = next((c for c in ["y_hat_lr_hi_95","y_hi_95","LinearRegression-hi-95","LR-hi-95"] if c in cols), None)

df_lr_std = (
    df_lr[["date", y_obs, y_hat] + ([lo, hi] if lo and hi else [])]
    .rename(columns={
        y_obs: "y_obs",
        y_hat: "LR",
        **({lo: "LR-lo-95", hi: "LR-hi-95"} if lo and hi else {})
    })
)

# =========================
# ---- Merge final ----
# =========================
df_all = (
    df_ar1_std
    .merge(df_arp_std, on="date", how="outer", suffixes=("", "_arp"))
    .merge(df_lr_std,  on="date", how="outer", suffixes=("", "_lr"))
    .sort_values("date")
)

# priorité y_obs : AR1 → ARp → LR
df_all["y_obs"] = (
    df_all["y_obs"]
    .combine_first(df_all.get("y_obs_arp"))
    .combine_first(df_all.get("y_obs_lr"))
)

# clean doublons
df_all = df_all[[c for c in df_all.columns if not c.endswith("_arp") and not c.endswith("_lr")]]

print(df_all.head())
print(df_all.columns.tolist())

        date  y_obs       AR1  AR1-lo-95  AR1-hi-95       ARp  ARp-lo-95  \
0 1963-12-01    0.0  0.070473        NaN        NaN  0.070473        NaN   
1 1964-01-01   -0.1  0.017682        NaN        NaN  0.017682        NaN   
2 1964-02-01   -0.5  0.090722        NaN        NaN  0.090722        NaN   
3 1964-03-01   -0.3  0.165681        NaN        NaN  0.165681        NaN   
4 1964-04-01   -0.4  0.091963        NaN        NaN  0.091963        NaN   

   ARp-hi-95        LR  LR-lo-95  LR-hi-95  
0        NaN -0.716835       NaN       NaN  
1        NaN -0.244223       NaN       NaN  
2        NaN  1.109324       NaN       NaN  
3        NaN  0.940781       NaN       NaN  
4        NaN  1.256649       NaN       NaN  
['date', 'y_obs', 'AR1', 'AR1-lo-95', 'AR1-hi-95', 'ARp', 'ARp-lo-95', 'ARp-hi-95', 'LR', 'LR-lo-95', 'LR-hi-95']


In [6]:
df_all = (
    df_ar1_std
    .merge(df_arp_std, on=["date", "y_obs"], how="outer")
    .merge(df_lr_std,  on=["date", "y_obs"], how="outer")
    .sort_values("date")
)

df_obs = (
    df_all.rename(columns={"date": "ds", "y_obs": "y"})
          .assign(unique_id=SERIES_ID)[["unique_id", "ds", "y"]]
)

# forecasts_df avec 3 modèles + intervalles si présents
fc_cols = [c for c in df_all.columns if c not in ("date", "y_obs")]
df_fcst = (
    df_all.rename(columns={"date": "ds"})
          .assign(unique_id=SERIES_ID)[["unique_id", "ds"] + fc_cols]
)

print(df_obs.head(2))
print(df_fcst.head(2))

  unique_id         ds    y
0    UNRATE 1963-12-01  0.0
1    UNRATE 1964-01-01 -0.1
  unique_id         ds       AR1  AR1-lo-95  AR1-hi-95       ARp  ARp-lo-95  \
0    UNRATE 1963-12-01  0.070473        NaN        NaN  0.070473        NaN   
1    UNRATE 1964-01-01  0.017682        NaN        NaN  0.017682        NaN   

   ARp-hi-95        LR  LR-lo-95  LR-hi-95  
0        NaN -0.716835       NaN       NaN  
1        NaN -0.244223       NaN       NaN  


In [8]:
START_ZOOM = "1990-01-01"
END_ZOOM   = "2025-08-01"

segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2008-01-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2019-fin"),
]

df_obs["ds"]  = pd.to_datetime(df_obs["ds"])
df_fcst["ds"] = pd.to_datetime(df_fcst["ds"])

start_zoom_dt = pd.to_datetime(START_ZOOM)
end_zoom_dt   = pd.to_datetime(END_ZOOM)

df_obs_z  = df_obs.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")
df_fcst_z = df_fcst.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")

fig = plot_series(
    df=df_obs_z,
    forecasts_df=df_fcst_z,
    level=[95],
    engine="plotly",
).update_layout(
    height=450,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
)

# Renommage legend
for trace in fig.data:
    name = (trace.name or "").lower()
    if trace.name == "y":
        trace.name = "Observed"
    elif trace.name == "AR1":
        trace.name = "AR(1) forecast"
    elif trace.name == "ARp":
        trace.name = "AR(p) forecast"
    elif trace.name == "LR":
        trace.name = "Linear Regression forecast"
    # intervalles: on laisse, sinon plotly peut mélanger

# Segments (sans 1990)
# ymin/ymax à partir des données dispo
vals_min = [df_obs_z["y"].min()]
vals_max = [df_obs_z["y"].max()]
for c in df_fcst_z.columns:
    if "lo-95" in c:
        vals_min.append(df_fcst_z[c].min())
    if "hi-95" in c:
        vals_max.append(df_fcst_z[c].max())
ymin = float(np.nanmin(vals_min))
ymax = float(np.nanmax(vals_max))

SEG_GROUP = "SEGMENTS"
fig.update_layout(legend=dict(groupclick="togglegroup"))

first = True
for i, (start, _, _) in enumerate(segments):
    if i == 0:
        continue
    x = pd.to_datetime(start)
    if not (start_zoom_dt <= x <= end_zoom_dt):
        continue
    fig.add_trace(
        go.Scatter(
            x=[x, x],
            y=[ymin, ymax],
            mode="lines",
            legendgroup=SEG_GROUP,
            name="Segments" if first else None,
            showlegend=first,
            visible="legendonly",
            line=dict(color="gray", width=1, dash="dash"),
            hoverinfo="skip",
        )
    )
    first = False

fig.show()

# Analyse de l'erreur

In [11]:
# =========================
# Construire df_pred_long depuis df_all
# =========================

dfs = []

# AR1
if "AR1" in df_all.columns:
    dfs.append(
        df_all[["date", "y_obs", "AR1"]]
        .rename(columns={"y_obs": "true", "AR1": "pred"})
        .assign(method="AR1")
    )

# ARp
if "ARp" in df_all.columns:
    dfs.append(
        df_all[["date", "y_obs", "ARp"]]
        .rename(columns={"y_obs": "true", "ARp": "pred"})
        .assign(method="ARP")
    )

# Linear Regression
if "LR" in df_all.columns:
    dfs.append(
        df_all[["date", "y_obs", "LR"]]
        .rename(columns={"y_obs": "true", "LR": "pred"})
        .assign(method="LINREG")
    )

df_pred_long = (
    pd.concat(dfs, ignore_index=True)
      .dropna(subset=["pred", "true"])
      .sort_values(["date", "method"])
)

print(df_pred_long.head())
print(df_pred_long["method"].value_counts())

           date  true      pred  method
0    1963-12-01   0.0  0.070473     AR1
741  1963-12-01   0.0  0.070473     ARP
1482 1963-12-01   0.0 -0.716835  LINREG
1    1964-01-01  -0.1  0.017682     AR1
742  1964-01-01  -0.1  0.017682     ARP
method
AR1       741
ARP       741
LINREG    741
Name: count, dtype: int64


In [12]:
# =========================
# Cellule 1 — Fonctions (MAE + DM)
# =========================
import numpy as np
import pandas as pd
from typing import Iterable, Optional, Tuple, List, Dict
from math import sqrt, erf, isfinite

# ----------------------------
# 0) Utilitaires
# ----------------------------
def _ensure_wide(df_wide: pd.DataFrame) -> pd.DataFrame:
    wide = df_wide.copy()
    if "date" in wide.columns:
        wide["date"] = pd.to_datetime(wide["date"], errors="coerce")
        wide = wide.set_index("date")
    if not isinstance(wide.index, pd.DatetimeIndex):
        raise ValueError("df_wide doit avoir un DatetimeIndex ou une colonne 'date' convertible en datetime.")
    wide = wide.sort_index()
    if "true" not in wide.columns:
        raise ValueError("df_wide doit contenir la colonne 'true'.")
    return wide

def _resolve_methods(wide: pd.DataFrame, methods: Optional[Iterable[str]]) -> List[str]:
    if methods is None:
        return [c for c in wide.columns if c != "true"]
    return [m for m in methods if m in wide.columns and m != "true"]

def _build_windows(
    wide: pd.DataFrame,
    periods: List[Tuple[str, Optional[str], str]],
    *,
    include_overall: bool = True,
    overall_label: str = "Ensemble",
) -> List[Tuple[pd.Timestamp, pd.Timestamp, str]]:
    full_start, full_end = wide.index.min(), wide.index.max()
    windows: List[Tuple[pd.Timestamp, pd.Timestamp, str]] = []
    if include_overall:
        windows.append((full_start, full_end, overall_label))
    for start, end, label in periods:
        s = pd.to_datetime(start)
        e = pd.to_datetime(end) if end is not None else full_end
        windows.append((s, e, label))
    return windows

# ----------------------------
# 1) MAE & erreurs par fenêtre
# ----------------------------
def _mae_and_errors_for_window(
    sub: pd.DataFrame,
    methods: List[str],
    *,
    min_obs: int = 20,
) -> Tuple[Dict[str, float], Dict[str, pd.Series]]:
    maes: Dict[str, float] = {}
    err_abs: Dict[str, pd.Series] = {}
    for m in methods:
        diffs = (sub["true"] - sub[m]).abs()
        valid = diffs.dropna()
        err_abs[m] = valid
        maes[m] = float(valid.mean()) if valid.shape[0] >= min_obs else np.nan
    return maes, err_abs

# ----------------------------
# 2) Diebold–Mariano (sans SciPy)
# ----------------------------
def _phi(z: float) -> float:
    return 0.5 * (1.0 + erf(z / sqrt(2.0)))

def _dm_pvalue(diff: np.ndarray, lags: int = 0) -> float:
    x = np.asarray(diff, dtype=float)
    x = x[np.isfinite(x)]
    T = x.size
    if T < 3:
        return np.nan

    dbar = x.mean()
    gamma0 = np.dot(x - dbar, x - dbar) / T
    var = gamma0

    if lags > 0:
        for k in range(1, min(lags, T - 1) + 1):
            w = 1.0 - k / (lags + 1.0)
            cov = np.dot(x[k:] - dbar, x[:-k] - dbar) / T
            var += 2.0 * w * cov

    if var <= 0:
        return np.nan

    stat = dbar / sqrt(var / T)
    p = 2.0 * (1.0 - _phi(abs(stat)))
    return max(0.0, min(1.0, p))

# ----------------------------
# 3) Formatage "MAE (p)"
# ----------------------------
def _format_cells_for_window(
    sub: pd.DataFrame,
    methods: List[str],
    maes: Dict[str, float],
    err_abs: Dict[str, pd.Series],
    *,
    min_obs: int,
    add_dm: bool,
    dm_lags: int,
    show_p_for_best: bool,
    round_digits: int,
    label_map: Optional[Dict[str, str]] = None,
) -> List[Tuple[str, str]]:
    finite_models = [m for m in methods if isfinite(maes.get(m, np.nan))]
    best_m = min(finite_models, key=lambda m: maes[m]) if finite_models else None

    out: List[Tuple[str, str]] = []
    for m in methods:
        label = label_map.get(m, m) if label_map else m
        mae_val = maes.get(m, np.nan)

        if not isfinite(mae_val):
            out.append((label, np.nan))
            continue

        p_txt = ""
        if add_dm and best_m is not None and m != best_m:
            v1 = err_abs[m]
            v2 = err_abs[best_m]
            common = v1.index.intersection(v2.index)
            diff = (v1.loc[common] - v2.loc[common]).values
            if diff.size >= min_obs:
                pval = _dm_pvalue(diff, lags=dm_lags)
                if isfinite(pval):
                    p_txt = f" ({pval:.3f})"
        elif add_dm and show_p_for_best and m == best_m:
            p_txt = " (—)"

        cell = f"{mae_val:.{round_digits}f}{p_txt}"
        out.append((label, cell))
    return out

# ----------------------------
# 4) Pivot final
# ----------------------------
def make_mae_dm_pivot(
    df_wide: pd.DataFrame,
    periods: List[Tuple[str, Optional[str], str]],
    *,
    methods: Optional[Iterable[str]] = None,
    include_overall: bool = True,
    overall_label: str = "Ensemble",
    min_obs: int = 20,
    round_digits: int = 4,
    add_dm: bool = True,
    dm_lags: int = 11,
    show_p_for_best: bool = False,
    label_map: Optional[Dict[str, str]] = None,
) -> pd.DataFrame:
    wide = _ensure_wide(df_wide)
    meths = _resolve_methods(wide, methods)
    if len(meths) == 0:
        return pd.DataFrame()

    windows = _build_windows(wide, periods, include_overall=include_overall, overall_label=overall_label)
    rows = []

    for start, end, label in windows:
        sub = wide.loc[start:end, ["true"] + meths].copy()
        sub = sub.dropna(subset=["true"])
        if len(sub) < min_obs:
            for m in meths:
                model_label = label_map.get(m, m) if label_map else m
                rows.append((model_label, label, np.nan))
            continue

        maes, err_abs = _mae_and_errors_for_window(sub, meths, min_obs=min_obs)

        pairs = _format_cells_for_window(
            sub=sub,
            methods=meths,
            maes=maes,
            err_abs=err_abs,
            min_obs=min_obs,
            add_dm=add_dm,
            dm_lags=dm_lags,
            show_p_for_best=show_p_for_best,
            round_digits=round_digits,
            label_map=label_map,
        )
        for model_label, cell in pairs:
            rows.append((model_label, label, cell))

    df_fmt = pd.DataFrame(rows, columns=["model", "period", "MAE_fmt"])

    desired_cols = ([overall_label] if include_overall else []) + [lbl for _, _, lbl in periods]
    pivot = df_fmt.pivot(index="model", columns="period", values="MAE_fmt").reindex(columns=desired_cols)

    order_index = [label_map.get(m, m) if label_map else m for m in meths]
    pivot = pivot.reindex(index=order_index)

    return pivot

In [22]:
# =========================
# Cellule 3 — Construction wide + tableau MAE/DM (AR1, ARP, LINREG)
# =========================

# Prérequis: df_pred_long doit exister et contenir au moins:
#   date | true | pred | method
df_tmp = df_pred_long.copy()
df_tmp["date"] = pd.to_datetime(df_tmp["date"], errors="coerce")

# (Optionnel) Vérifier les méthodes existantes
print("Méthodes dispo :", sorted(df_tmp["method"].dropna().unique()))

# True par date (identique pour tous les modèles)
true_by_date = (
    df_tmp.dropna(subset=["true"])
          .groupby("date")["true"]
          .first()
          .rename("true")
)

# Pivot wide: colonnes = méthodes, index = date
wide = (
    df_tmp.pivot_table(index="date", columns="method", values="pred", aggfunc="mean")
          .sort_index()
          .join(true_by_date, how="left")
)

# Méthodes à évaluer
methods_keep = ["AR1", "ARP", "LINREG"]
methods_keep = [m for m in methods_keep if m in wide.columns]

# Tableau MAE (p) par segments
table_pivot = make_mae_dm_pivot(
    df_wide=wide,
    periods=segments,
    methods=methods_keep,
    include_overall=True,
    overall_label="Ensemble",
    min_obs=20,
    round_digits=4,
    add_dm=True,
    dm_lags=11,     # h-1 si h=12
    show_p_for_best=False,
)

print("Méthodes utilisées :", methods_keep)

# =========================
# 🎨 Styling (affichage)
# =========================
def highlight_best(s):
    vals = s.str.extract(r"([0-9.]+)")[0].astype(float)
    if vals.isna().all():
        return [""] * len(s)
    best = vals.min()
    return [
        "font-weight:bold; color:#2ca02c" if v == best else ""
        for v in vals
    ]

styled_table = (
    table_pivot.style
    .apply(highlight_best, axis=0)
    .set_table_styles([
        # En-têtes colonnes
        {"selector": "th.col_heading",
         "props": [("background-color", "#f0f0f0"),
                   ("color", "#000000"),
                   ("font-weight", "bold")]},

        # Nom de l'index ("model")
        {"selector": "th.index_name",
         "props": [("background-color", "#f0f0f0"),
                   ("color", "#000000"),
                   ("font-weight", "bold"),
                   ("text-align", "left")]},

        # 🔑 NOMS DES MODÈLES (AR1 / ARP / LINREG)
        {"selector": "th.row_heading",
         "props": [
             ("background-color", "#f0f0f0"),
             ("color", "#000000"),
             ("font-weight", "bold"),
             ("font-size", "13px"),
             ("text-align", "left"),
         ]},

        # Cellules
        {"selector": "td",
         "props": [("padding", "6px")]},
    ])
)

display(styled_table)

Méthodes dispo : ['AR1', 'ARP', 'LINREG']
Méthodes utilisées : ['AR1', 'ARP', 'LINREG']


period,Ensemble,1990-1999,2000-2008,2008-2019,2019-fin
model,,,,,
AR1,0.8788 (0.138),0.5078 (0.109),0.5356 (0.437),0.8686 (0.604),2.0848 (0.138)
ARP,0.8779 (0.163),0.4951,0.5287 (0.492),0.8442 (0.776),2.1595 (0.157)
LINREG,0.8163,0.5399 (0.512),0.4745,0.8107,1.9171
